In [13]:
import sys
import warnings
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
from scipy.stats import norm

sys.path.insert(0, "../../Utilities/")
import common_functions as cf
warnings.filterwarnings('ignore')

In [14]:
def ucb(x, gp, kappa = 2.0):
    # Get the mean (mu) and std (sigma) from the GP
    mu, sigma = gp.predict(x, return_std=True)
    return mu + kappa * sigma

In [15]:
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy')
    
print(X)
print(Y)

[[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.56682913]]
[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837]


In [17]:
X_init = np.array([
 [0.17152521, 0.34391687, 0.2487372 ],
 [0.24211446, 0.64407427, 0.27243281],
 [0.53490572, 0.39850092, 0.17338873],
 [0.49258141, 0.61159319, 0.34017639],
 [0.13462167, 0.21991724, 0.45820622],
 [0.34552327, 0.94135983, 0.26936348],
 [0.15183663, 0.43999062, 0.99088187],
 [0.64550284, 0.39714294, 0.91977134],
 [0.74691195, 0.28419631, 0.22629985],
 [0.17047699, 0.6970324 , 0.14916943],
 [0.22054934, 0.29782524, 0.34355534],
 [0.66601366, 0.67198515, 0.2462953 ],
 [0.04680895, 0.23136024, 0.77061759],
 [0.60009728, 0.72513573, 0.06608864],
 [0.96599485, 0.86111969, 0.56682913]]
)

y_init = np.array(
 [-0.1121222,  -0.08796286, -0.11141465, -0.03483531, -0.04800758, -0.11062091,
 -0.39892551, -0.11386851, -0.13146061, -0.09418956, -0.04694741, -0.10596504,
 -0.11804826, -0.03637783, -0.05675837]
)

#Gaussian Process
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

#acquistion function
def acquisition(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu = mu[0]
    sigma = sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei

y_best = y_init.max()
bounds = [(0,1), (0,1), (0,1)]

best_x = None
best_ei = float('inf')
for _ in range(10):
    x0 = np.random.rand(3)
    res = minimize(lambda x: acquisition(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print(cf.format_inputdata(x_next))


0.876550-0.378323-0.422209
